#  **Project : Unified Military Analytics and Comparison Dashboard**


## **MILESTONE 2 : KPI Engineering and Tableau Prep**

##  **Module 3: KPI Feature Engineering**

**Internship:** Infosys Springboard  
**Domain:** Data Visualization (DV)  
**Author:** Yogeshwar Vadla

**Mentor: Sirisha (Ma'am)**

This notebook implements **Module 3** of the project, focusing on **deriving
high-impact Key Performance Indicators (KPIs)** and enriching the cleaned
military dataset with **geopolitical metadata** to enable meaningful analysis.




##  **Module Overview**

In this module, raw military metrics are transformed into **normalized,
comparison-ready indicators** such as Power Index Rank Gap, Assets per Capita,
and Budget-to-GDP Ratio. The dataset is further enhanced with **region,
continent, and alliance (NATO) flags**, ensuring it is fully optimized for
**Power BI / Tableau dashboards**.

The final output is delivered in both **wide and long analytical formats**,
allowing seamless KPI comparisons, ranking analysis, and interactive
visual storytelling in the final dashboard.

## **Objectives**

- Compute core military KPIs:
  - **Power Index Rank Gap**
  - **Assets per Capita**
  - **Budget-to-GDP Ratio**
- Enrich data with geopolitical metadata:
  - Continent
  - Region
  - Alliance flags (NATO / Non-NATO)
- Prepare datasets in **wide** and **long** formats
- Export a consolidated analytics-ready Excel file

## **Load Cleaned Data**

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/Colab Notebooks/Unified Military Analytics and Comparison Dashboard-DV/Milestone 1/module 2/data/military_cleaned.csv"
df = pd.read_csv(path)
print("Dataset loaded successfully")
print("Total rows:", df.shape[0])

Dataset loaded successfully
Total rows: 145


## **Numeric conversion**

Converting Required Columns to Numeric. Ensure Required Columns Are Numeric

In [ ]:
numeric_columns = [
    "Power_Index",
    "total_population",
    "defense_budget_usd",
    "purchasing_power_parity_usd",
    "total_military_aircraft",
    "total_military_helicopters",
    "tanks",
    "total_naval_fleet"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversion completed")

df.dtypes


Numeric conversion completed


,0
Country_Full_Name,object
Rank,int64
Country_Short_Name,object
Power_Index,float64
total_population,int64
total_military_manpower,int64
fit_for_service,int64
population_reaching_military_age_annually,int64
active_personnel,int64
reserve_personnel,int64


##  **KPI Definitions**


### **KPI 1: Power Index Rank Gap**

**Formula:**  
` Power_Index_Rank_Gap = Power_Index_Rank − Rank  `



**Purpose:**  
Identifies countries that **over-perform or under-perform** relative to their rank.


In [ ]:
# Lower Power_Index = stronger military
df["Power_Index_Rank"] = df["Power_Index"].rank(ascending=True, method="min")

# KPI 1: Power Index Rank Gap
df["Power_Index_Rank_Gap"] = df["Power_Index_Rank"] - df["Rank"]

df[[
    "Country_Full_Name",
    "Rank",
    "Power_Index",
    "Power_Index_Rank",
    "Power_Index_Rank_Gap"
]].head()


,Country_Full_Name,Rank,Power_Index,Power_Index_Rank,Power_Index_Rank_Gap
0,United States,1,0.0744,1.0,0.0
1,Russia,2,0.0788,2.0,0.0
2,China,3,0.0788,2.0,-1.0
3,India,4,0.1184,4.0,0.0
4,South Korea,5,0.1656,5.0,0.0


### **KPI 2: Assets per Capita**

**Formula:**  
`Assets per Capita = Total Assets / Total Population
`
```
Total_Military_Assets = (Total Aircraft + Total Helicopters + Tanks + Naval Fleet) / Total Population
```

**Purpose:**  
Normalizes military assets by population to enable fair cross-country comparison.

In [ ]:
df["Total_Military_Assets"] = (
    df["total_military_aircraft"]
    + df["total_military_helicopters"]
    + df["tanks"]
    + df["total_naval_fleet"]
)

df["Assets_per_Capita"] = (
    df["Total_Military_Assets"] / df["total_population"].replace(0, pd.NA)
)

df[["Country_Full_Name", "Assets_per_Capita"]].head()


,Country_Full_Name,Assets_per_Capita
0,United States,0.000070
1,Russia,0.000086
2,China,0.000008
3,India,0.000005
4,South Korea,0.000093


### **KPI 3: Budget-to-GDP Ratio**

**Formula:**  
`Defense Budget / Purchasing Power Parity (GDP)`


**Purpose:**  
Measures the proportion of economic output allocated to defense spending.


In [ ]:
df["Budget_to_GDP_Ratio"] = (
    df["defense_budget_usd"] / df["purchasing_power_parity_usd"].replace(0, pd.NA)
)

df[["Country_Full_Name", "Budget_to_GDP_Ratio"]].head()


,Country_Full_Name,Budget_to_GDP_Ratio
0,United States,0.036291
1,Russia,0.021664
2,China,0.008545
3,India,0.005723
4,South Korea,0.017706


##  **Metadata Enrichment**

To support geopolitical and alliance-based analysis, the dataset is enriched with:

- **Continent**
- **Region**
- **NATO Membership Flag**

This enables insights such as:
- NATO vs Non-NATO military comparison
- Regional defense concentration
- Continental power distribution

Example:

```
metadata = {
    "United States": ["North America", "North America", "NATO"],
    "India": ["South Asia", "Asia", "Non-NATO"],
    "China": ["East Asia", "Asia", "Non-NATO"]
}

```

In [ ]:
continent_map = {
    "United States": "North America",
    "Canada": "North America",
    "India": "Asia",
    "China": "Asia",
    "Russia": "Europe/Asia",
    "United Kingdom": "Europe",
    "France": "Europe",
    "Germany": "Europe"
}

region_map = {
    "United States": "North America",
    "Canada": "North America",
    "India": "South Asia",
    "China": "East Asia",
    "Russia": "Eastern Europe",
    "United Kingdom": "Western Europe",
    "France": "Western Europe",
    "Germany": "Western Europe"
}

nato_countries = [
    "United States", "Canada", "United Kingdom", "France",
    "Germany", "Italy", "Spain", "Turkey", "Poland", "Netherlands"
]


In [ ]:
df["Continent"] = df["Country_Full_Name"].map(continent_map)
df["Region"] = df["Country_Full_Name"].map(region_map)

df["Is_NATO_Member"] = df["Country_Full_Name"].isin(nato_countries)

df[["Country_Full_Name", "Continent", "Region", "Is_NATO_Member"]].head()


,Country_Full_Name,Continent,Region,Is_NATO_Member
0,United States,North America,North America,True
1,Russia,Europe/Asia,Eastern Europe,False
2,China,Asia,East Asia,False
3,India,Asia,South Asia,False
4,South Korea,NaN,NaN,False


In [ ]:
df["Continent"].fillna("Other", inplace=True)
df["Region"].fillna("Other", inplace=True)


/tmp/ipython-input-1588517439.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Continent"].fillna("Other", inplace=True)
/tmp/ipython-input-1588517439.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

## **Output & Deliverables**

**File:** `military_final.xlsx`

**Sheets Included:**
- **wide_format** – One row per country with all KPIs and metrics
- **long_format** – KPI-centric structure for dynamic visualizations

The dataset is fully compatible with **Power BI and Tableau**.


## **wide & Long Format**

Used for KPI comparisons & dynamic charts.

In [ ]:
final_columns = [

    # Identity
    "Country_Full_Name",
    "Country_Short_Name",

    # Metadata
    "Continent",
    "Region",
    "Is_NATO_Member",

    # Power Index
    "Power_Index",
    "Power_Index_Rank",
    "Power_Index_Rank_Gap",

    # KPIs
    "Assets_per_Capita",
    "Budget_to_GDP_Ratio",

    # Manpower
    "total_population",
    "total_military_manpower",
    "active_personnel",
    "reserve_personnel",
    "paramilitary",

    # Air Power
    "total_military_aircraft",
    "fighter_aircraft",
    "attack_aircraft",
    "total_military_helicopters",

    # Land Power
    "tanks",
    "armored_fighting_vehicles",
    "self_propelled_artillery",
    "rocket_projectors",

    # Naval Power
    "total_naval_fleet",
    "aircraft_carriers",
    "submarines",
    "destroyers",
    "frigates",

    # Economy & Infra
    "defense_budget_usd",
    "total_serviceable_airports",
    "major_ports_and_terminals",
    "total_land_area_sq_km"
]

final_df = df[final_columns]
final_df.head()


,Country_Full_Name,Country_Short_Name,Continent,Region,Is_NATO_Member,Power_Index,Power_Index_Rank,Power_Index_Rank_Gap,Assets_per_Capita,Budget_to_GDP_Ratio,...,rocket_projectors,total_naval_fleet,aircraft_carriers,submarines,destroyers,frigates,defense_budget_usd,total_serviceable_airports,major_ports_and_terminals,total_land_area_sq_km
0,United States,USA,North America,North America,True,0.0744,1.0,0.0,0.000070,0.036291,...,641,440,11,70,81,0,895000000000,15873,666,9833517
1,Russia,RUS,Europe/Asia,Eastern Europe,False,0.0788,2.0,0.0,0.000086,0.021664,...,3005,419,1,63,10,12,126000000000,904,67,17098242
2,China,CHN,Asia,East Asia,False,0.0788,2.0,-1.0,0.000008,0.008545,...,2750,754,3,61,50,47,266850000000,531,66,9596960
3,India,IND,Asia,South Asia,False,0.1184,4.0,0.0,0.000005,0.005723,...,264,293,2,18,13,14,75000000000,311,56,3287263
4,South Korea,SKO,Other,Other,False,0.1656,5.0,0.0,0.000093,0.017706,...,426,227,0,22,13,17,46300000000,89,15,99720


In [ ]:
final_df.dtypes

,0
Country_Full_Name,object
Country_Short_Name,object
Continent,object
Region,object
Is_NATO_Member,bool
Power_Index,float64
Power_Index_Rank,float64
Power_Index_Rank_Gap,float64
Assets_per_Capita,float64
Budget_to_GDP_Ratio,float64


In [ ]:
kpi_columns = [
    "Power_Index_Rank_Gap",
    "Assets_per_Capita",
    "Budget_to_GDP_Ratio"
]

long_df = pd.melt(
    final_df,
    id_vars=[
        "Country_Full_Name",
        "Country_Short_Name",
        "Continent",
        "Region",
        "Is_NATO_Member"
    ],
    value_vars=kpi_columns,
    var_name="KPI_Name",
    value_name="KPI_Value"
)

long_df.head()


,Country_Full_Name,Country_Short_Name,Continent,Region,Is_NATO_Member,KPI_Name,KPI_Value
0,United States,USA,North America,North America,True,Power_Index_Rank_Gap,0.0
1,Russia,RUS,Europe/Asia,Eastern Europe,False,Power_Index_Rank_Gap,0.0
2,China,CHN,Asia,East Asia,False,Power_Index_Rank_Gap,-1.0
3,India,IND,Asia,South Asia,False,Power_Index_Rank_Gap,0.0
4,South Korea,SKO,Other,Other,False,Power_Index_Rank_Gap,0.0


In [ ]:
long_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country_Full_Name   435 non-null    object 
 1   Country_Short_Name  435 non-null    object 
 2   Continent           435 non-null    object 
 3   Region              435 non-null    object 
 4   Is_NATO_Member      435 non-null    bool   
 5   KPI_Name            435 non-null    object 
 6   KPI_Value           435 non-null    float64
dtypes: bool(1), float64(1), object(5)
memory usage: 20.9+ KB


## **Export Final Deliverable**
**File:** `military_final.xlsx`


In [ ]:
import sys
import os

# Install xlsxwriter if not already installed
!pip install xlsxwriter

output_dir = "/content/drive/MyDrive/Colab Notebooks/Unified Military Analytics and Comparison Dashboard-DV/Milestone 2/module_3/data/military_final.xlsx"

# # Create the directory if it doesn't exist
# os.makedirs(output_dir, exist_ok=True)

# output_path = os.path.join(output_dir, "military_final.xlsx")

with pd.ExcelWriter(
    # output_path,
    output_dir,
    engine="xlsxwriter"
) as writer:
    final_df.to_excel(writer, sheet_name="wide_format", index=False)
    long_df.to_excel(writer, sheet_name="long_format", index=False)

print("military_final.xlsx created successfully")
print("Total countries:", final_df.shape[0])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.9 MB/s eta 0:00:00
military_final.xlsx created successfully
Total countries: 145


## **Module 3 Summary**

### **Summary**
Module 3 successfully transforms the cleaned military dataset into an
**analytics-ready, insight-driven dataset** by engineering key performance
indicators (KPIs) and enriching the data with essential geopolitical metadata.
Three core KPIs—**Power Index Rank Gap**, **Assets per Capita**, and
**Budget-to-GDP Ratio**—were computed to enable fair, normalized comparison
across countries with varying population sizes and economic scales.

Additionally, the dataset was enriched with **continent, region, and NATO
alliance indicators**, providing crucial contextual dimensions for strategic
analysis. The final deliverable was exported in both **wide and long formats**,
ensuring seamless compatibility with **Power BI and Tableau** without any
additional transformations.

---

### **Key Insights**
- **Power Index Rank Gap** highlights discrepancies between official rankings
  and computed military strength, helping identify over- or under-ranked nations.
- **Assets per Capita** reveals true force density, offering a more realistic
  measure of military capacity relative to population size.
- **Budget-to-GDP Ratio** provides insight into a country’s defense prioritization
  relative to its economic strength.
- Metadata enrichment enables **regional, continental, and alliance-based
  comparisons**, supporting geopolitical and strategic storytelling in dashboards.
- The structured wide/long formats allow flexible KPI visualization, ranking,
  and trend analysis.

---

### **Next Steps**
- Integrate the final dataset into **Power BI dashboards** for interactive
  exploration and comparison.
- Develop **KPI-driven visuals** such as rank-gap heatmaps, per-capita asset
  comparisons, and budget-efficiency charts.
- Add advanced analytical layers such as **composite military strength scores**
  or **regional benchmarking indices**.
- Perform time-series analysis in future iterations by incorporating
  historical data for multi-year trend evaluation.
